# Chapter 3 — Structured Tools and Deterministic Batches

This chapter turns the asynchronous model Runtime into a complete structured Tool loop while preserving provider neutrality, deterministic history, and replaceable execution infrastructure.

## Goal and Previous Limitation

Chapter 2 can observe streamed Tool Calls but stops after returning them. It cannot validate arguments, execute a capability, return a structured result, or continue the model conversation. We begin by loading that immutable Checkpoint and making the missing round trip explicit.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_2 = ROOT / 'course' / 'checkpoints' / 'ch02'
sys.path.insert(0, str(CHAPTER_2 / 'src'))
import agent_harness as chapter2

assert hasattr(chapter2, 'AgentRuntime')
assert not hasattr(chapter2, 'Tool')
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]
sys.path.pop(0)


## Conceptual Model

`AgentRuntime` remains the deep module: callers still use `start(messages)` or `run(messages)`. The interface adds only explicit `tools`, a replaceable `tool_executor`, and an output budget. Runtime hides JSON parsing, Draft 2020-12 Schema validation, the preflight seam, batch scheduling, exception normalization, truncation, result ordering, and model continuation.

A `Tool` owns the model-visible declaration and local behavior. A `PreparedToolCall` is the stable value after ordered preflight and before execution, so later Hooks can intercept one clear seam. `ToolExecutor` is a Protocol with a local adapter by default; remote or isolated adapters can replace it without changing Tool declarations or the Agent Loop.

## Minimal Execution

The next Export Cells replace the evolved model and Runtime modules, add the Tool module, and publish the complete Chapter 3 interface. Each cell owns a complete source file.

In [ ]:
MODEL_SOURCE = r'''"""Provider-neutral message and scripted model contracts for Chapter 3 work."""

from __future__ import annotations

from collections.abc import AsyncIterator, Mapping, Sequence
from dataclasses import dataclass, field
from enum import Enum
from types import MappingProxyType
from typing import Any, Literal, Protocol, TypeAlias, cast
from urllib.parse import urlparse

import openai


class Role(str, Enum):
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"


class StopReason(str, Enum):
    COMPLETE = "complete"
    TOOL_USE = "tool_use"
    LENGTH = "length"
    CONTENT_FILTER = "content_filter"
    ERROR = "error"
    ABORTED = "aborted"
    OTHER = "other"


class ModelErrorCode(str, Enum):
    AUTHENTICATION = "authentication"
    REQUEST = "request"
    SCHEMA = "schema"
    RATE_LIMIT = "rate_limit"
    TIMEOUT = "timeout"
    CONNECTION = "connection"
    SERVER = "server"
    PROVIDER = "provider"


class UnsupportedContentError(ValueError):
    """Raised before provider I/O for unsupported content."""


class ModelProtocolError(RuntimeError):
    """Raised when an adapter violates the provider-neutral stream contract."""


@dataclass(frozen=True, slots=True)
class TextContent:
    text: str
    type: Literal["text"] = field(default="text", init=False)
    schema_version: Literal[1] = field(default=1, init=False)

    def __post_init__(self) -> None:
        if not isinstance(self.text, str):
            raise TypeError("TextContent.text must be a string")


@dataclass(frozen=True, slots=True)
class ToolCallContent:
    id: str
    name: str
    arguments: str
    type: Literal["tool_call"] = field(default="tool_call", init=False)
    schema_version: Literal[1] = field(default=1, init=False)

    def __post_init__(self) -> None:
        if not self.id or not self.name:
            raise ValueError("a Tool Call requires non-empty id and name")
        if not isinstance(self.arguments, str):
            raise TypeError("ToolCallContent.arguments must be a JSON string")


ContentBlock: TypeAlias = TextContent | ToolCallContent


@dataclass(frozen=True, slots=True)
class ModelMessage:
    role: Role
    content: tuple[ContentBlock, ...]


@dataclass(frozen=True, slots=True)
class ModelToolResultMessage:
    tool_call_id: str
    tool_name: str
    content: str
    is_error: bool = False
    role: Literal[Role.TOOL] = field(default=Role.TOOL, init=False)


ModelContextMessage: TypeAlias = ModelMessage | ModelToolResultMessage


@dataclass(frozen=True, slots=True)
class AgentMessage:
    role: Role
    content: tuple[ContentBlock, ...]

    @classmethod
    def text(cls, role: Role, text: str) -> AgentMessage:
        return cls(role=role, content=(TextContent(text),))

    def to_model(self) -> ModelMessage:
        if self.role is Role.TOOL:
            raise UnsupportedContentError("Tool results require ToolResultMessage")
        content = tuple(_validate_content(block, self.role) for block in self.content)
        return ModelMessage(self.role, content)


@dataclass(frozen=True, slots=True)
class ModelSpec:
    model_id: str
    context_window: int | None = None
    max_output_tokens: int = 4096
    supports_tools: bool = True

    def __post_init__(self) -> None:
        if not self.model_id.strip():
            raise ValueError("ModelSpec.model_id cannot be empty")
        if self.context_window is not None and self.context_window <= 0:
            raise ValueError("context_window must be positive when supplied")
        if self.max_output_tokens <= 0:
            raise ValueError("max_output_tokens must be positive")


@dataclass(frozen=True, slots=True)
class Usage:
    input_tokens: int
    output_tokens: int
    total_tokens: int
    estimated: bool = False


@dataclass(frozen=True, slots=True)
class ToolDefinition:
    name: str
    description: str
    input_schema: Mapping[str, object]

    def __post_init__(self) -> None:
        object.__setattr__(self, "input_schema", MappingProxyType(dict(self.input_schema)))


@dataclass(frozen=True, slots=True)
class ModelRequest:
    messages: tuple[ModelContextMessage, ...]
    model: ModelSpec
    tools: tuple[ToolDefinition, ...] = ()


@dataclass(frozen=True, slots=True)
class ModelResult:
    message: ModelMessage
    stop_reason: StopReason
    usage: Usage | None = None


@dataclass(frozen=True, slots=True)
class ModelError:
    code: ModelErrorCode
    message: str
    retryable: bool
    status_code: int | None = None
    retry_after_seconds: float | None = None


class ModelAdapterError(RuntimeError):
    def __init__(self, error: ModelError) -> None:
        self.error = error
        super().__init__(error.message)


@dataclass(frozen=True, slots=True)
class TextDelta:
    text: str


@dataclass(frozen=True, slots=True)
class ToolCallDelta:
    index: int
    id: str = ""
    name: str = ""
    arguments_delta: str = ""


@dataclass(frozen=True, slots=True)
class UsageUpdate:
    usage: Usage


@dataclass(frozen=True, slots=True)
class ModelEnd:
    stop_reason: StopReason


ModelEvent: TypeAlias = TextDelta | ToolCallDelta | UsageUpdate | ModelEnd


class ModelAdapter(Protocol):
    def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]: ...


def _validate_content(block: object, role: Role) -> ContentBlock:
    if not isinstance(block, (TextContent, ToolCallContent)):
        raise UnsupportedContentError(
            "version one supports only TextContent and ToolCallContent"
        )
    if getattr(block, "schema_version", None) != 1:
        raise UnsupportedContentError("unsupported Content Block schema version")
    if isinstance(block, ToolCallContent) and role is not Role.ASSISTANT:
        raise UnsupportedContentError(
            "ToolCallContent is valid only for assistant messages"
        )
    return block


def to_model_messages(messages: Sequence[object]) -> tuple[ModelContextMessage, ...]:
    converted: list[ModelContextMessage] = []
    for message in messages:
        convert = getattr(message, "to_model", None)
        if not callable(convert):
            raise TypeError("conversation messages must provide to_model()")
        converted.append(convert())
    return tuple(converted)


class ScriptedModelAdapter:
    """Replay one or more provider-neutral model turns."""

    def __init__(
        self,
        events: Sequence[ModelEvent] | Sequence[Sequence[ModelEvent]],
    ) -> None:
        items = tuple(events)
        if items and isinstance(items[0], (list, tuple)):
            self._turns = tuple(tuple(turn) for turn in items)  # type: ignore[arg-type]
        else:
            self._turns = (items,)  # type: ignore[assignment]
        self._requests: list[ModelRequest] = []

    @property
    def received_requests(self) -> tuple[ModelRequest, ...]:
        return tuple(self._requests)

    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
        turn = len(self._requests)
        self._requests.append(request)
        if turn >= len(self._turns):
            raise RuntimeError("script has no model turn remaining")
        for event in self._turns[turn]:
            yield event


async def complete(
    adapter: ModelAdapter,
    messages: Sequence[AgentMessage],
    model: ModelSpec,
) -> ModelResult:
    request = ModelRequest(to_model_messages(messages), model)
    text_parts: list[str] = []
    tool_drafts: dict[int, dict[str, str]] = {}
    usage: Usage | None = None
    end: ModelEnd | None = None
    async for event in adapter.stream(request):
        if end is not None:
            raise ModelProtocolError("an adapter emitted data after ModelEnd")
        if isinstance(event, TextDelta):
            text_parts.append(event.text)
        elif isinstance(event, ToolCallDelta):
            if event.index < 0:
                raise ModelProtocolError("Tool Call indexes cannot be negative")
            draft = tool_drafts.setdefault(
                event.index, {"id": "", "name": "", "arguments": ""}
            )
            draft["id"] += event.id
            draft["name"] += event.name
            draft["arguments"] += event.arguments_delta
        elif isinstance(event, UsageUpdate):
            usage = event.usage
        elif isinstance(event, ModelEnd):
            end = event
        else:
            raise ModelProtocolError(
                f"unsupported model event: {type(event).__name__}"
            )
    if end is None:
        raise ModelProtocolError("an adapter stream must end with ModelEnd")
    blocks: list[ContentBlock] = []
    if text_parts:
        blocks.append(TextContent("".join(text_parts)))
    for index in sorted(tool_drafts):
        draft = tool_drafts[index]
        try:
            blocks.append(ToolCallContent(**draft))
        except (TypeError, ValueError) as error:
            raise ModelProtocolError(f"incomplete Tool Call at index {index}") from error
    return ModelResult(
        message=ModelMessage(Role.ASSISTANT, tuple(blocks)),
        stop_reason=end.stop_reason,
        usage=usage,
    )


@dataclass(frozen=True, slots=True)
class OpenAICompatibleConfig:
    base_url: str
    api_key: str
    headers: Mapping[str, str] = field(default_factory=dict)
    extra_body: Mapping[str, object] = field(default_factory=dict)
    timeout_seconds: float = 60.0

    def __post_init__(self) -> None:
        parsed = urlparse(self.base_url)
        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            raise ValueError("base_url must be an explicit HTTP(S) URL")
        if not self.api_key:
            raise ValueError("api_key must be supplied explicitly")
        if self.timeout_seconds <= 0:
            raise ValueError("timeout_seconds must be positive")
        object.__setattr__(self, "headers", MappingProxyType(dict(self.headers)))
        object.__setattr__(self, "extra_body", MappingProxyType(dict(self.extra_body)))


def _provider_message(message: ModelContextMessage) -> dict[str, object]:
    if isinstance(message, ModelToolResultMessage):
        return {
            "role": "tool",
            "tool_call_id": message.tool_call_id,
            "content": message.content,
        }
    text = "".join(
        block.text for block in message.content if isinstance(block, TextContent)
    )
    tool_calls = [
        block for block in message.content if isinstance(block, ToolCallContent)
    ]
    encoded: dict[str, object] = {
        "role": message.role.value,
        "content": text or None,
    }
    if tool_calls:
        encoded["tool_calls"] = [
            {
                "id": block.id,
                "type": "function",
                "function": {"name": block.name, "arguments": block.arguments},
            }
            for block in tool_calls
        ]
    return encoded


def _provider_tool(tool: ToolDefinition) -> dict[str, object]:
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": dict(tool.input_schema),
        },
    }


def _stop_reason(value: str | None) -> StopReason:
    if value is None:
        return StopReason.OTHER
    return {
        "stop": StopReason.COMPLETE,
        "tool_calls": StopReason.TOOL_USE,
        "length": StopReason.LENGTH,
        "content_filter": StopReason.CONTENT_FILTER,
    }.get(value, StopReason.OTHER)


def _normalized_error(error: Exception) -> ModelError:
    status = getattr(error, "status_code", None)
    retry_after_seconds: float | None = None
    response = getattr(error, "response", None)
    headers = getattr(response, "headers", None)
    if headers is not None:
        raw_retry_after = headers.get("retry-after")
        if raw_retry_after is not None:
            try:
                parsed_retry_after = float(raw_retry_after)
            except (TypeError, ValueError):
                pass
            else:
                if parsed_retry_after >= 0:
                    retry_after_seconds = parsed_retry_after
    if isinstance(error, openai.AuthenticationError):
        code, retryable = ModelErrorCode.AUTHENTICATION, False
    elif isinstance(error, openai.RateLimitError) or status == 429:
        code, retryable = ModelErrorCode.RATE_LIMIT, True
    elif isinstance(error, openai.APITimeoutError) or status == 408:
        code, retryable = ModelErrorCode.TIMEOUT, True
    elif isinstance(error, openai.APIConnectionError):
        code, retryable = ModelErrorCode.CONNECTION, True
    elif isinstance(status, int) and status >= 500:
        code, retryable = ModelErrorCode.SERVER, True
    elif isinstance(error, (openai.BadRequestError, openai.NotFoundError)):
        code, retryable = ModelErrorCode.REQUEST, False
    else:
        code, retryable = ModelErrorCode.PROVIDER, False
    status_text = f" with status {status}" if status is not None else ""
    return ModelError(
        code=code,
        message=f"OpenAI-compatible request failed{status_text}",
        retryable=retryable,
        status_code=status,
        retry_after_seconds=retry_after_seconds,
    )


class OpenAICompatibleAdapter:
    """Translate streaming Chat Completions at the ModelAdapter seam."""

    def __init__(self, config: OpenAICompatibleConfig) -> None:
        self._config = config
        self._client = openai.AsyncOpenAI(
            api_key=config.api_key,
            base_url=config.base_url.rstrip("/") + "/",
            default_headers=dict(config.headers),
            timeout=config.timeout_seconds,
            max_retries=0,
        )

    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
        finish_reason: str | None = None
        try:
            response = await self._client.chat.completions.create(
                model=request.model.model_id,
                messages=cast(list, [_provider_message(item) for item in request.messages]),
                max_tokens=request.model.max_output_tokens,
                tools=cast(
                    Any,
                    [_provider_tool(tool) for tool in request.tools]
                    if request.tools
                    else openai.NOT_GIVEN,
                ),
                stream=True,
                stream_options={"include_usage": True},
                extra_body=dict(self._config.extra_body) or None,
            )
            async for chunk in response:
                if chunk.usage is not None:
                    input_tokens = chunk.usage.prompt_tokens or 0
                    output_tokens = chunk.usage.completion_tokens or 0
                    total_tokens = (
                        chunk.usage.total_tokens or input_tokens + output_tokens
                    )
                    yield UsageUpdate(
                        Usage(input_tokens, output_tokens, total_tokens)
                    )
                for choice in chunk.choices:
                    delta = choice.delta
                    if delta.content:
                        yield TextDelta(delta.content)
                    for tool_call in delta.tool_calls or ():
                        function = tool_call.function
                        yield ToolCallDelta(
                            index=tool_call.index,
                            id=tool_call.id or "",
                            name=(function.name if function else None) or "",
                            arguments_delta=(
                                function.arguments if function else None
                            )
                            or "",
                        )
                    if choice.finish_reason is not None:
                        finish_reason = choice.finish_reason
        except openai.OpenAIError as error:
            raise ModelAdapterError(_normalized_error(error)) from None
        yield ModelEnd(_stop_reason(finish_reason))
'''


In [ ]:
TOOLS_SOURCE = r'''"""Explicit Tools and their replaceable execution seam."""

from __future__ import annotations

from collections.abc import Awaitable, Callable, Mapping
from dataclasses import dataclass, field
from enum import Enum
from types import MappingProxyType
from typing import Protocol

from jsonschema import Draft202012Validator  # type: ignore[import-untyped]

from .model import ModelToolResultMessage, ToolCallContent, ToolDefinition


ToolHandler = Callable[[dict[str, object]], Awaitable["ToolResult"]]


class ToolErrorCode(str, Enum):
    INVALID_JSON = "invalid_json"
    INVALID_ARGUMENTS = "invalid_arguments"
    UNKNOWN_TOOL = "unknown_tool"
    EXECUTION_FAILED = "execution_failed"


class TruncationDirection(str, Enum):
    HEAD = "head"
    TAIL = "tail"


class CompleteOutputKind(str, Enum):
    ARTIFACT = "artifact"
    EXTERNAL = "external"
    UNAVAILABLE = "unavailable"


@dataclass(frozen=True, slots=True)
class CompleteOutputReference:
    kind: CompleteOutputKind
    reference: str | None = None
    reason: str | None = None

    @classmethod
    def artifact(cls, reference: str) -> "CompleteOutputReference":
        return cls(CompleteOutputKind.ARTIFACT, reference=reference)

    @classmethod
    def unavailable(cls) -> "CompleteOutputReference":
        return cls(
            CompleteOutputKind.UNAVAILABLE,
            reason="complete output was not retained",
        )

    def render(self) -> str:
        if self.reference is not None:
            return f"{self.kind.value} {self.reference}"
        return f"{self.kind.value} ({self.reason})"


@dataclass(frozen=True, slots=True)
class TruncationNotice:
    original_bytes: int
    original_lines: int
    retained_start_byte: int
    retained_end_byte: int
    retained_start_line: int
    retained_end_line: int
    direction: TruncationDirection


@dataclass(frozen=True, slots=True)
class ToolOutputBudget:
    max_bytes: int = 50 * 1024
    max_lines: int = 2000

    def __post_init__(self) -> None:
        if self.max_bytes <= 0 or self.max_lines <= 0:
            raise ValueError("Tool output limits must be positive")


@dataclass(frozen=True, slots=True)
class ToolResult:
    content: str
    metadata: Mapping[str, object] = field(default_factory=dict)
    terminate: bool = False
    is_error: bool = False
    error_code: ToolErrorCode | None = None
    truncation: TruncationNotice | None = None
    complete_output: CompleteOutputReference | None = None

    def __post_init__(self) -> None:
        object.__setattr__(self, "metadata", MappingProxyType(dict(self.metadata)))
        if self.is_error != (self.error_code is not None):
            raise ValueError("error ToolResult values require exactly one error_code")
        if self.is_error and self.terminate:
            raise ValueError("error ToolResult values cannot terminate a Tool Batch")

    @classmethod
    def error(cls, code: ToolErrorCode, tool_name: str) -> "ToolResult":
        guidance = {
            ToolErrorCode.INVALID_JSON: "provide one valid JSON object",
            ToolErrorCode.INVALID_ARGUMENTS: "match the Tool's declared input schema",
            ToolErrorCode.UNKNOWN_TOOL: "choose one of the advertised Tools",
            ToolErrorCode.EXECUTION_FAILED: "revise the call or choose another Tool",
        }[code]
        return cls(
            content=f"Tool error [{code.value}] for '{tool_name}': {guidance}.",
            is_error=True,
            error_code=code,
        )


@dataclass(frozen=True, slots=True)
class Tool:
    name: str
    description: str
    input_schema: Mapping[str, object]
    execute: ToolHandler
    sequential: bool = False
    output_direction: TruncationDirection = TruncationDirection.HEAD

    def __post_init__(self) -> None:
        if not self.name.strip():
            raise ValueError("Tool name cannot be empty")
        if not self.description.strip():
            raise ValueError("Tool description cannot be empty")
        if not callable(self.execute):
            raise TypeError("Tool execute must be an async callable")
        schema = dict(self.input_schema)
        Draft202012Validator.check_schema(schema)
        if schema.get("type") != "object":
            raise ValueError("Tool input_schema must describe a JSON object")
        object.__setattr__(self, "input_schema", MappingProxyType(schema))

    def definition(self) -> ToolDefinition:
        return ToolDefinition(self.name, self.description, self.input_schema)


@dataclass(frozen=True, slots=True)
class PreparedToolCall:
    call: ToolCallContent
    tool: Tool
    arguments: dict[str, object]


@dataclass(frozen=True, slots=True)
class ToolResultMessage:
    tool_call_id: str
    tool_name: str
    result: ToolResult

    def to_model(self) -> ModelToolResultMessage:
        return ModelToolResultMessage(
            self.tool_call_id,
            self.tool_name,
            self.result.content,
            self.result.is_error,
        )


class ToolExecutor(Protocol):
    async def execute(self, call: PreparedToolCall) -> ToolResult: ...


class LocalToolExecutor:
    async def execute(self, call: PreparedToolCall) -> ToolResult:
        return await call.tool.execute(call.arguments)


def bound_tool_result(
    result: ToolResult,
    budget: ToolOutputBudget,
    direction: TruncationDirection,
) -> ToolResult:
    """Bound model-facing text and attach explicit recovery provenance."""

    content = result.content
    encoded = content.encode("utf-8")
    lines = content.splitlines(keepends=True)
    original_lines = len(content.splitlines())
    if len(encoded) <= budget.max_bytes and original_lines <= budget.max_lines:
        return result

    if direction is TruncationDirection.HEAD:
        line_limited = "".join(lines[: budget.max_lines])
        retained_bytes = line_limited.encode("utf-8")[: budget.max_bytes]
        retained = retained_bytes.decode("utf-8", errors="ignore")
        retained_start_byte = 0
        retained_end_byte = len(retained.encode("utf-8"))
        retained_start_line = 1 if retained else 0
        retained_end_line = len(retained.splitlines())
    else:
        line_limited = "".join(lines[-budget.max_lines :])
        retained_bytes = line_limited.encode("utf-8")[-budget.max_bytes :]
        retained = retained_bytes.decode("utf-8", errors="ignore")
        retained_end_byte = len(encoded)
        retained_start_byte = retained_end_byte - len(retained.encode("utf-8"))
        retained_end_line = original_lines
        retained_line_count = len(retained.splitlines())
        retained_start_line = max(1, original_lines - retained_line_count + 1)

    reference = result.complete_output or CompleteOutputReference.unavailable()
    notice = TruncationNotice(
        original_bytes=len(encoded),
        original_lines=original_lines,
        retained_start_byte=retained_start_byte,
        retained_end_byte=retained_end_byte,
        retained_start_line=retained_start_line,
        retained_end_line=retained_end_line,
        direction=direction,
    )
    model_notice = (
        "[tool output truncated: "
        f"retained {direction.value} bytes {retained_start_byte}-{retained_end_byte} "
        f"of {len(encoded)}, lines {retained_start_line}-{retained_end_line} "
        f"of {original_lines}; complete output: {reference.render()}]"
    )
    return ToolResult(
        content=f"{retained}\n\n{model_notice}",
        metadata=result.metadata,
        terminate=result.terminate,
        is_error=result.is_error,
        error_code=result.error_code,
        truncation=notice,
        complete_output=reference,
    )
'''


In [ ]:
RUNTIME_SOURCE = r'''"""Async Agent Runtime with structured Tool batches."""

from __future__ import annotations

import asyncio
from collections.abc import AsyncIterator, Awaitable, Callable, Sequence
from dataclasses import dataclass
from enum import Enum
import json
from typing import TypeAlias, cast

from jsonschema import (  # type: ignore[import-untyped]
    Draft202012Validator,
    ValidationError,
)

from .model import (
    AgentMessage,
    ContentBlock,
    ModelAdapter,
    ModelAdapterError,
    ModelEnd,
    ModelError,
    ModelErrorCode,
    ModelEvent,
    ModelRequest,
    ModelSpec,
    Role,
    StopReason,
    TextContent,
    TextDelta,
    ToolCallContent,
    ToolCallDelta,
    Usage,
    UsageUpdate,
    to_model_messages,
)
from .tools import (
    LocalToolExecutor,
    PreparedToolCall,
    Tool,
    ToolErrorCode,
    ToolExecutor,
    ToolOutputBudget,
    ToolResult,
    ToolResultMessage,
    bound_tool_result,
)


class EventType(str, Enum):
    AGENT_START = "agent_start"
    MODEL_ATTEMPT_START = "model_attempt_start"
    MODEL_EVENT = "model_event"
    MODEL_ATTEMPT_FAILED = "model_attempt_failed"
    RETRY_SCHEDULED = "retry_scheduled"
    TOOL_BATCH_START = "tool_batch_start"
    TOOL_CALL_START = "tool_call_start"
    TOOL_CALL_END = "tool_call_end"
    TOOL_BATCH_END = "tool_batch_end"
    RUN_CANCELLED = "run_cancelled"
    MESSAGE_END = "message_end"
    AGENT_END = "agent_end"


@dataclass(frozen=True, slots=True)
class RuntimeEvent:
    sequence: int
    type: EventType
    attempt: int | None = None
    model_event: ModelEvent | None = None
    error: ModelError | None = None
    retry_delay_seconds: float | None = None
    partial_text: str = ""
    partial_usage: Usage | None = None
    tool_call_id: str | None = None
    tool_name: str | None = None
    tool_result: ToolResult | None = None


@dataclass(frozen=True, slots=True)
class AssistantOutcome:
    message: AgentMessage
    stop_reason: StopReason
    usage: Usage | None = None
    error: ModelError | None = None
    attempts: int = 1
    tool_results: tuple[ToolResult, ...] = ()


@dataclass(frozen=True, slots=True)
class RetryPolicy:
    delays: tuple[float, ...] = (2.0, 4.0, 8.0)
    max_retry_after_seconds: float = 60.0

    def __post_init__(self) -> None:
        if any(delay < 0 for delay in self.delays):
            raise ValueError("retry delays cannot be negative")
        if self.max_retry_after_seconds < 0:
            raise ValueError("max_retry_after_seconds cannot be negative")

    def delay_for(
        self,
        error: ModelError,
        failed_attempt: int,
        *,
        retry_after_seconds: float | None = None,
    ) -> float | None:
        retryable_codes = {
            ModelErrorCode.RATE_LIMIT,
            ModelErrorCode.TIMEOUT,
            ModelErrorCode.CONNECTION,
            ModelErrorCode.SERVER,
        }
        retryable_status = error.status_code in {408, 429} or (
            error.status_code is not None and error.status_code >= 500
        )
        if (
            error.code not in retryable_codes and not retryable_status
        ) or failed_attempt > len(self.delays):
            return None
        if retry_after_seconds is not None:
            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:
                return None
            return retry_after_seconds
        return self.delays[failed_attempt - 1]


Sleeper: TypeAlias = Callable[[float], Awaitable[None]]
ConversationMessage: TypeAlias = AgentMessage | ToolResultMessage
_EVENTS_DONE = object()


def _add_usage(left: Usage | None, right: Usage | None) -> Usage | None:
    if left is None:
        return right
    if right is None:
        return left
    return Usage(
        left.input_tokens + right.input_tokens,
        left.output_tokens + right.output_tokens,
        left.total_tokens + right.total_tokens,
        left.estimated or right.estimated,
    )


class AgentRunHandle:
    """One accepted run's observations, cancellation, and eventual outcome."""

    def __init__(
        self,
        task: asyncio.Task[AssistantOutcome],
        events: asyncio.Queue[RuntimeEvent | object],
    ) -> None:
        self._task = task
        self._events = events
        self._cancel_requested = False

    async def events(self) -> AsyncIterator[RuntimeEvent]:
        while True:
            event = await self._events.get()
            if event is _EVENTS_DONE:
                break
            yield cast(RuntimeEvent, event)

    async def result(self) -> AssistantOutcome:
        return await self._task

    def cancel(self) -> None:
        if not self._cancel_requested and not self._task.done():
            self._cancel_requested = True
            self._task.get_loop().call_soon(self._task.cancel)


class AgentRuntime:
    """Advance typed conversation state through model and Tool turns."""

    def __init__(
        self,
        adapter: ModelAdapter,
        model: ModelSpec,
        *,
        tools: Sequence[Tool] = (),
        tool_executor: ToolExecutor | None = None,
        tool_output_budget: ToolOutputBudget | None = None,
        retry_policy: RetryPolicy | None = None,
        sleeper: Sleeper = asyncio.sleep,
        run_guard: object | None = None,
    ) -> None:
        if not isinstance(model, ModelSpec):
            raise TypeError("model must be a ModelSpec")
        if not callable(getattr(adapter, "stream", None)):
            raise TypeError("adapter must implement ModelAdapter.stream")
        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):
            raise TypeError("retry_policy must be a RetryPolicy")
        if not callable(sleeper):
            raise TypeError("sleeper must be an async callable")
        if tool_executor is not None and not callable(
            getattr(tool_executor, "execute", None)
        ):
            raise TypeError("tool_executor must implement ToolExecutor.execute")
        if tool_output_budget is not None and not isinstance(
            tool_output_budget, ToolOutputBudget
        ):
            raise TypeError("tool_output_budget must be a ToolOutputBudget")
        registered: dict[str, Tool] = {}
        for tool in tools:
            if not isinstance(tool, Tool):
                raise TypeError("tools must contain Tool values")
            if tool.name in registered:
                raise ValueError(f"duplicate Tool name: {tool.name!r}")
            registered[tool.name] = tool
        if registered and not model.supports_tools:
            raise ValueError("configured ModelSpec does not support Tools")
        self._adapter = adapter
        self._model = model
        self._tools = registered
        self._tool_executor = tool_executor or LocalToolExecutor()
        self._tool_output_budget = tool_output_budget or ToolOutputBudget()
        self._retry_policy = retry_policy or RetryPolicy()
        self._sleeper = sleeper
        self._run_guard = run_guard
        self._history: list[ConversationMessage] = []

    @property
    def history(self) -> tuple[ConversationMessage, ...]:
        return tuple(self._history)

    @property
    def run_guard(self) -> object | None:
        return self._run_guard

    def start(self, messages: Sequence[AgentMessage]) -> AgentRunHandle:
        accepted = tuple(messages)
        to_model_messages((*self._history, *accepted))
        self._history.extend(accepted)
        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()
        task = asyncio.get_running_loop().create_task(self._execute(events))
        return AgentRunHandle(task, events)

    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:
        return await self.start(messages).result()

    async def _execute(
        self,
        events: asyncio.Queue[RuntimeEvent | object],
    ) -> AssistantOutcome:
        sequence = 0
        total_attempts = 0
        text_parts: list[str] = []
        current_usage: Usage | None = None
        run_usage: Usage | None = None
        run_tool_results: list[ToolResult] = []

        async def emit(
            type_: EventType,
            *,
            attempt: int | None = None,
            model_event: ModelEvent | None = None,
            error: ModelError | None = None,
            retry_delay_seconds: float | None = None,
            partial_text: str = "",
            partial_usage: Usage | None = None,
            tool_call_id: str | None = None,
            tool_name: str | None = None,
            tool_result: ToolResult | None = None,
        ) -> None:
            nonlocal sequence
            sequence += 1
            await events.put(
                RuntimeEvent(
                    sequence=sequence,
                    type=type_,
                    attempt=attempt,
                    model_event=model_event,
                    error=error,
                    retry_delay_seconds=retry_delay_seconds,
                    partial_text=partial_text,
                    partial_usage=partial_usage,
                    tool_call_id=tool_call_id,
                    tool_name=tool_name,
                    tool_result=tool_result,
                )
            )

        async def finish(outcome: AssistantOutcome) -> AssistantOutcome:
            self._history.append(outcome.message)
            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))
            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))
            return outcome

        try:
            await emit(EventType.AGENT_START)
            while True:
                turn_attempt = 0
                while True:
                    turn_attempt += 1
                    total_attempts += 1
                    attempt = total_attempts
                    request = ModelRequest(
                        to_model_messages(self._history),
                        self._model,
                        tuple(tool.definition() for tool in self._tools.values()),
                    )
                    await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)
                    text_parts = []
                    tool_drafts: dict[int, dict[str, str]] = {}
                    current_usage = None
                    end: ModelEnd | None = None
                    schema_error: ModelError | None = None
                    try:
                        async for event in self._adapter.stream(request):
                            await emit(
                                EventType.MODEL_EVENT,
                                attempt=attempt,
                                model_event=event,
                            )
                            if end is not None:
                                schema_error = ModelError(
                                    ModelErrorCode.SCHEMA,
                                    "model stream emitted data after ModelEnd",
                                    False,
                                )
                                break
                            if isinstance(event, TextDelta):
                                text_parts.append(event.text)
                            elif isinstance(event, ToolCallDelta):
                                if event.index < 0:
                                    schema_error = ModelError(
                                        ModelErrorCode.SCHEMA,
                                        "model stream emitted an invalid Tool Call index",
                                        False,
                                    )
                                    break
                                draft = tool_drafts.setdefault(
                                    event.index,
                                    {"id": "", "name": "", "arguments": ""},
                                )
                                draft["id"] += event.id
                                draft["name"] += event.name
                                draft["arguments"] += event.arguments_delta
                            elif isinstance(event, UsageUpdate):
                                current_usage = event.usage
                            elif isinstance(event, ModelEnd):
                                end = event
                            else:
                                schema_error = ModelError(
                                    ModelErrorCode.SCHEMA,
                                    "model stream emitted an unsupported event",
                                    False,
                                )
                                break
                    except ModelAdapterError as failure:
                        partial_text = "".join(text_parts)
                        await emit(
                            EventType.MODEL_ATTEMPT_FAILED,
                            attempt=attempt,
                            error=failure.error,
                            partial_text=partial_text,
                            partial_usage=current_usage,
                        )
                        delay = self._retry_policy.delay_for(
                            failure.error,
                            turn_attempt,
                            retry_after_seconds=failure.error.retry_after_seconds,
                        )
                        if delay is not None:
                            await emit(
                                EventType.RETRY_SCHEDULED,
                                attempt=attempt,
                                error=failure.error,
                                retry_delay_seconds=delay,
                                partial_text=partial_text,
                                partial_usage=current_usage,
                            )
                            text_parts = []
                            current_usage = None
                            await self._sleeper(delay)
                            continue
                        terminal_usage = _add_usage(run_usage, current_usage)
                        return await finish(
                            AssistantOutcome(
                                AgentMessage.text(Role.ASSISTANT, partial_text),
                                StopReason.ERROR,
                                terminal_usage,
                                failure.error,
                                total_attempts,
                                tuple(run_tool_results),
                            )
                        )

                    error = schema_error
                    if error is None and end is None:
                        error = ModelError(
                            ModelErrorCode.SCHEMA,
                            "model stream violated the provider-neutral event contract",
                            False,
                        )
                    blocks: list[ContentBlock] = []
                    if error is None:
                        try:
                            if text_parts:
                                blocks.append(TextContent("".join(text_parts)))
                            for index in sorted(tool_drafts):
                                blocks.append(ToolCallContent(**tool_drafts[index]))
                        except (TypeError, ValueError):
                            error = ModelError(
                                ModelErrorCode.SCHEMA,
                                "model stream emitted an incomplete Tool Call",
                                False,
                            )
                    if error is not None:
                        partial_text = "".join(text_parts)
                        await emit(
                            EventType.MODEL_ATTEMPT_FAILED,
                            attempt=attempt,
                            error=error,
                            partial_text=partial_text,
                            partial_usage=current_usage,
                        )
                        terminal_usage = _add_usage(run_usage, current_usage)
                        return await finish(
                            AssistantOutcome(
                                AgentMessage.text(Role.ASSISTANT, partial_text),
                                StopReason.ERROR,
                                terminal_usage,
                                error,
                                total_attempts,
                                tuple(run_tool_results),
                            )
                        )
                    assert end is not None
                    break

                run_usage = _add_usage(run_usage, current_usage)
                assistant = AgentMessage(Role.ASSISTANT, tuple(blocks))
                self._history.append(assistant)
                await emit(EventType.MESSAGE_END, attempt=total_attempts)
                calls = tuple(
                    block
                    for block in assistant.content
                    if isinstance(block, ToolCallContent)
                )
                if not calls:
                    await emit(EventType.AGENT_END, attempt=total_attempts)
                    return AssistantOutcome(
                        assistant,
                        end.stop_reason,
                        run_usage,
                        attempts=total_attempts,
                        tool_results=tuple(run_tool_results),
                    )

                await emit(EventType.TOOL_BATCH_START, attempt=total_attempts)
                prepared: dict[int, PreparedToolCall] = {}
                results: dict[int, ToolResult] = {}
                for index, call in enumerate(calls):
                    tool = self._tools.get(call.name)
                    if tool is None:
                        results[index] = ToolResult.error(
                            ToolErrorCode.UNKNOWN_TOOL, call.name
                        )
                        continue
                    try:
                        parsed = json.loads(call.arguments)
                    except (json.JSONDecodeError, TypeError):
                        results[index] = ToolResult.error(
                            ToolErrorCode.INVALID_JSON, call.name
                        )
                        continue
                    try:
                        Draft202012Validator(tool.input_schema).validate(parsed)
                    except ValidationError:
                        results[index] = ToolResult.error(
                            ToolErrorCode.INVALID_ARGUMENTS, call.name
                        )
                        continue
                    if not isinstance(parsed, dict):
                        results[index] = ToolResult.error(
                            ToolErrorCode.INVALID_ARGUMENTS, call.name
                        )
                        continue
                    prepared[index] = PreparedToolCall(call, tool, parsed)

                async def execute_one(index: int, call: PreparedToolCall) -> None:
                    await emit(
                        EventType.TOOL_CALL_START,
                        attempt=total_attempts,
                        tool_call_id=call.call.id,
                        tool_name=call.call.name,
                    )
                    try:
                        result = await self._tool_executor.execute(call)
                        if not isinstance(result, ToolResult):
                            raise TypeError("ToolExecutor returned an invalid result")
                    except Exception:
                        result = ToolResult.error(
                            ToolErrorCode.EXECUTION_FAILED, call.call.name
                        )
                    result = bound_tool_result(
                        result,
                        self._tool_output_budget,
                        call.tool.output_direction,
                    )
                    results[index] = result
                    await emit(
                        EventType.TOOL_CALL_END,
                        attempt=total_attempts,
                        tool_call_id=call.call.id,
                        tool_name=call.call.name,
                        tool_result=result,
                    )

                if any(call.tool.sequential for call in prepared.values()):
                    for index, prepared_call in prepared.items():
                        await execute_one(index, prepared_call)
                else:
                    await asyncio.gather(
                        *(
                            execute_one(index, prepared_call)
                            for index, prepared_call in prepared.items()
                        )
                    )

                batch_results: list[ToolResult] = []
                for index, call in enumerate(calls):
                    result = results[index]
                    batch_results.append(result)
                    run_tool_results.append(result)
                    self._history.append(
                        ToolResultMessage(call.id, call.name, result)
                    )
                await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)
                if batch_results and all(result.terminate for result in batch_results):
                    await emit(EventType.AGENT_END, attempt=total_attempts)
                    return AssistantOutcome(
                        assistant,
                        end.stop_reason,
                        run_usage,
                        attempts=total_attempts,
                        tool_results=tuple(run_tool_results),
                    )
        except asyncio.CancelledError:
            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))
            outcome = AssistantOutcome(
                message,
                StopReason.ABORTED,
                _add_usage(run_usage, current_usage),
                attempts=max(total_attempts, 1),
                tool_results=tuple(run_tool_results),
            )
            self._history.append(message)
            await emit(
                EventType.RUN_CANCELLED,
                attempt=max(total_attempts, 1),
                partial_text="".join(text_parts),
                partial_usage=current_usage,
            )
            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))
            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))
            return outcome
        finally:
            await events.put(_EVENTS_DONE)
'''


In [ ]:
INIT_SOURCE = r'''from .model import (
    AgentMessage,
    ContentBlock,
    ModelAdapter,
    ModelAdapterError,
    ModelEnd,
    ModelError,
    ModelErrorCode,
    ModelEvent,
    ModelMessage,
    ModelProtocolError,
    ModelRequest,
    ModelResult,
    ModelSpec,
    ModelToolResultMessage,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    ScriptedModelAdapter,
    StopReason,
    TextContent,
    TextDelta,
    ToolCallContent,
    ToolCallDelta,
    UnsupportedContentError,
    Usage,
    UsageUpdate,
    complete,
    to_model_messages,
)
from .runtime import (
    AgentRunHandle,
    AgentRuntime,
    AssistantOutcome,
    EventType,
    RetryPolicy,
    RuntimeEvent,
    Sleeper,
)
from .tools import (
    CompleteOutputKind,
    CompleteOutputReference,
    LocalToolExecutor,
    PreparedToolCall,
    Tool,
    ToolErrorCode,
    ToolExecutor,
    ToolOutputBudget,
    ToolResult,
    ToolResultMessage,
    TruncationDirection,
    TruncationNotice,
)

__all__ = [name for name in globals() if not name.startswith("_")]
'''


In [ ]:
import asyncio
import importlib
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix='chapter-03-minimal-') as temporary:
    package = Path(temporary) / 'agent_harness'
    package.mkdir()
    (package / 'model.py').write_text(MODEL_SOURCE, encoding='utf-8')
    (package / 'tools.py').write_text(TOOLS_SOURCE, encoding='utf-8')
    (package / 'runtime.py').write_text(RUNTIME_SOURCE, encoding='utf-8')
    (package / '__init__.py').write_text(INIT_SOURCE, encoding='utf-8')
    sys.path.insert(0, temporary)
    try:
        chapter3 = importlib.import_module('agent_harness')

        async def minimal_tool_run():
            async def add(arguments):
                return chapter3.ToolResult(str(arguments['left'] + arguments['right']))

            adapter = chapter3.ScriptedModelAdapter([
                [
                    chapter3.ToolCallDelta(
                        0, 'call-add', 'add', '{"left": 2, "right": 3}'
                    ),
                    chapter3.ModelEnd(chapter3.StopReason.TOOL_USE),
                ],
                [
                    chapter3.TextDelta('The sum is 5.'),
                    chapter3.ModelEnd(chapter3.StopReason.COMPLETE),
                ],
            ])
            runtime = chapter3.AgentRuntime(
                adapter,
                chapter3.ModelSpec('scripted/chapter-03'),
                tools=[chapter3.Tool(
                    'add',
                    'Add two integers',
                    {
                        'type': 'object',
                        'properties': {
                            'left': {'type': 'integer'},
                            'right': {'type': 'integer'},
                        },
                        'required': ['left', 'right'],
                        'additionalProperties': False,
                    },
                    add,
                )],
            )
            handle = runtime.start([
                chapter3.AgentMessage.text(chapter3.Role.USER, 'add 2 and 3')
            ])
            outcome = await handle.result()
            events = [event async for event in handle.events()]
            return outcome, events, runtime.history, adapter.received_requests

        minimal_outcome, minimal_events, minimal_history, minimal_requests = (
            await minimal_tool_run()
        )
    finally:
        sys.path.pop(0)
        for module_name in tuple(sys.modules):
            if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
                del sys.modules[module_name]

assert minimal_outcome.message.content[0].text == 'The sum is 5.'
assert len(minimal_requests) == 2
assert any(isinstance(message, chapter3.ToolResultMessage) for message in minimal_history)


## Staged Construction

The first tracer bullet drove one typed Tool round trip through public interfaces. Successive red–green slices added safe errors, the replaceable executor, parallel reads with ordered persistence, sequential batches, bounded output, unanimous termination, and OpenAI-compatible Tool translation. The cumulative Chapter 2 regression suite stayed green before these sources entered the Notebook.

Preflight always follows model call order. If every prepared Tool is parallel-safe, Runtime starts them together and emits completion Events as they actually finish. If any prepared Tool is sequential, all executable calls run in source order. Final `ToolResultMessage` values are always materialized in source order.

In [ ]:
TOOL_TEST_SOURCE = r'''from __future__ import annotations

import asyncio

import pytest
from jsonschema.exceptions import SchemaError

from agent_harness import (
    AgentMessage,
    AgentRuntime,
    CompleteOutputReference,
    EventType,
    ModelEnd,
    ModelSpec,
    PreparedToolCall,
    Role,
    ScriptedModelAdapter,
    StopReason,
    TextDelta,
    Tool,
    ToolCallDelta,
    ToolErrorCode,
    ToolExecutor,
    ToolOutputBudget,
    ToolResult,
    ToolResultMessage,
)


def test_model_can_call_one_typed_tool_and_continue() -> None:
    async def scenario() -> None:
        received: list[dict[str, object]] = []

        async def add(arguments: dict[str, object]) -> ToolResult:
            received.append(arguments)
            return ToolResult(content=str(arguments["left"] + arguments["right"]))

        adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "call-add", "add", '{"left": 2, "right": 3}'),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("The sum is 5."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/tools"),
            tools=[
                Tool(
                    name="add",
                    description="Add two integers",
                    input_schema={
                        "type": "object",
                        "properties": {
                            "left": {"type": "integer"},
                            "right": {"type": "integer"},
                        },
                        "required": ["left", "right"],
                        "additionalProperties": False,
                    },
                    execute=add,
                )
            ],
        )

        outcome = await runtime.run([AgentMessage.text(Role.USER, "add 2 and 3")])

        assert outcome.stop_reason is StopReason.COMPLETE
        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "The sum is 5.")
        assert received == [{"left": 2, "right": 3}]
        result_messages = [
            message for message in runtime.history if isinstance(message, ToolResultMessage)
        ]
        assert len(result_messages) == 1
        assert result_messages[0].tool_call_id == "call-add"
        assert result_messages[0].result.content == "5"
        assert adapter.received_requests[1].messages[-1] == result_messages[0].to_model()

    asyncio.run(scenario())


def test_tool_declarations_are_explicit_and_runtime_registers_nothing_implicitly() -> None:
    async def execute(arguments: dict[str, object]) -> ToolResult:
        return ToolResult("ok")

    with pytest.raises(ValueError, match="name"):
        Tool("", "description", {"type": "object"}, execute)
    with pytest.raises(ValueError, match="description"):
        Tool("named", "", {"type": "object"}, execute)
    with pytest.raises(SchemaError):
        Tool("broken", "Bad schema", {"type": "not-a-json-schema-type"}, execute)
    with pytest.raises(ValueError, match="cannot terminate"):
        ToolResult(
            "failed",
            terminate=True,
            is_error=True,
            error_code=ToolErrorCode.EXECUTION_FAILED,
        )

    async def scenario() -> None:
        adapter = ScriptedModelAdapter(
            [TextDelta("No Tools installed."), ModelEnd(StopReason.COMPLETE)]
        )
        runtime = AgentRuntime(adapter, ModelSpec("scripted/no-tools"))

        await runtime.run([AgentMessage.text(Role.USER, "plain completion")])

        assert adapter.received_requests[0].tools == ()

    asyncio.run(scenario())


def test_oversized_tool_output_is_bounded_with_notice_and_complete_reference() -> None:
    async def scenario() -> None:
        async def verbose(arguments: dict[str, object]) -> ToolResult:
            return ToolResult(
                "abcdefghijk",
                complete_output=CompleteOutputReference.artifact("sha256:full-output"),
            )

        adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "verbose-call", "verbose", "{}"),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("Output was bounded."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/output-budget"),
            tools=[
                Tool(
                    "verbose",
                    "Return verbose output",
                    {"type": "object"},
                    verbose,
                )
            ],
            tool_output_budget=ToolOutputBudget(max_bytes=5, max_lines=10),
        )

        await runtime.run([AgentMessage.text(Role.USER, "be verbose")])

        message = next(
            item for item in runtime.history if isinstance(item, ToolResultMessage)
        )
        assert message.result.truncation is not None
        assert message.result.truncation.original_bytes == 11
        assert message.result.truncation.retained_start_byte == 0
        assert message.result.truncation.retained_end_byte == 5
        assert message.result.complete_output == CompleteOutputReference.artifact(
            "sha256:full-output"
        )
        assert message.result.content.startswith("abcde\n\n[tool output truncated:")
        assert "complete output: artifact sha256:full-output" in message.result.content
        assert adapter.received_requests[1].messages[-1].content == message.result.content

    asyncio.run(scenario())


def test_read_only_batch_finishes_in_parallel_but_history_keeps_call_order() -> None:
    async def scenario() -> None:
        second_started = asyncio.Event()

        async def first(arguments: dict[str, object]) -> ToolResult:
            await second_started.wait()
            await asyncio.sleep(0)
            return ToolResult("first result")

        async def second(arguments: dict[str, object]) -> ToolResult:
            second_started.set()
            return ToolResult("second result")

        schema = {"type": "object", "additionalProperties": False}
        adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "first-call", "first", "{}"),
                    ToolCallDelta(1, "second-call", "second", "{}"),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("Both complete."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/parallel"),
            tools=[
                Tool("first", "First read", schema, first),
                Tool("second", "Second read", schema, second),
            ],
        )

        handle = runtime.start([AgentMessage.text(Role.USER, "read both")])
        events_task = asyncio.create_task(
            _collect_events(handle)
        )
        outcome = await asyncio.wait_for(handle.result(), timeout=1)
        events = await events_task

        assert outcome.stop_reason is StopReason.COMPLETE
        completed_ids = [
            event.tool_call_id
            for event in events
            if event.type is EventType.TOOL_CALL_END
        ]
        assert completed_ids == ["second-call", "first-call"]
        persisted_ids = [
            message.tool_call_id
            for message in runtime.history
            if isinstance(message, ToolResultMessage)
        ]
        assert persisted_ids == ["first-call", "second-call"]

    async def _collect_events(handle):
        return [event async for event in handle.events()]

    asyncio.run(scenario())


def test_one_sequential_tool_forces_the_whole_batch_into_source_order() -> None:
    async def scenario() -> None:
        active = 0
        maximum_active = 0
        activity: list[str] = []

        def handler(name: str):
            async def execute(arguments: dict[str, object]) -> ToolResult:
                nonlocal active, maximum_active
                active += 1
                maximum_active = max(maximum_active, active)
                activity.append(f"start:{name}")
                await asyncio.sleep(0)
                activity.append(f"end:{name}")
                active -= 1
                return ToolResult(name)

            return execute

        schema = {"type": "object"}
        adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "one", "one", "{}"),
                    ToolCallDelta(1, "write", "write", "{}"),
                    ToolCallDelta(2, "three", "three", "{}"),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("Sequential batch complete."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/sequential"),
            tools=[
                Tool("one", "Read one", schema, handler("one")),
                Tool("write", "Mutate state", schema, handler("write"), sequential=True),
                Tool("three", "Read three", schema, handler("three")),
            ],
        )

        await runtime.run([AgentMessage.text(Role.USER, "run the batch")])

        assert maximum_active == 1
        assert activity == [
            "start:one",
            "end:one",
            "start:write",
            "end:write",
            "start:three",
            "end:three",
        ]

    asyncio.run(scenario())


def test_model_continuation_is_skipped_only_for_unanimous_termination() -> None:
    async def scenario() -> None:
        async def stop(arguments: dict[str, object]) -> ToolResult:
            return ToolResult("stop", terminate=True)

        async def keep_going(arguments: dict[str, object]) -> ToolResult:
            return ToolResult("continue", terminate=False)

        schema = {"type": "object"}
        tools = [
            Tool("stop", "Terminate", schema, stop),
            Tool("keep_going", "Continue", schema, keep_going),
        ]
        unanimous_adapter = ScriptedModelAdapter(
            [
                ToolCallDelta(0, "stop-one", "stop", "{}"),
                ToolCallDelta(1, "stop-two", "stop", "{}"),
                ModelEnd(StopReason.TOOL_USE),
            ]
        )
        unanimous_runtime = AgentRuntime(
            unanimous_adapter,
            ModelSpec("scripted/unanimous"),
            tools=tools,
        )

        unanimous = await unanimous_runtime.run(
            [AgentMessage.text(Role.USER, "both stop")]
        )

        assert len(unanimous_adapter.received_requests) == 1
        assert unanimous.stop_reason is StopReason.TOOL_USE
        assert [result.terminate for result in unanimous.tool_results] == [True, True]

        mixed_adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "mixed-stop", "stop", "{}"),
                    ToolCallDelta(1, "mixed-go", "keep_going", "{}"),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("Mixed batch continued."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        mixed_runtime = AgentRuntime(
            mixed_adapter,
            ModelSpec("scripted/mixed"),
            tools=tools,
        )

        mixed = await mixed_runtime.run([AgentMessage.text(Role.USER, "mixed batch")])

        assert len(mixed_adapter.received_requests) == 2
        assert mixed.stop_reason is StopReason.COMPLETE
        assert [result.terminate for result in mixed.tool_results] == [True, False]

    asyncio.run(scenario())


def test_replacing_tool_executor_changes_backend_without_changing_loop() -> None:
    async def scenario() -> None:
        local_handler_called = False

        async def local_handler(arguments: dict[str, object]) -> ToolResult:
            nonlocal local_handler_called
            local_handler_called = True
            return ToolResult("local")

        class RemoteExecutor:
            def __init__(self) -> None:
                self.calls: list[PreparedToolCall] = []

            async def execute(self, call: PreparedToolCall) -> ToolResult:
                self.calls.append(call)
                return ToolResult("remote:ok", metadata={"backend": "remote"})

        executor: ToolExecutor = RemoteExecutor()
        adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "call-remote", "lookup", '{"key": "answer"}'),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("Remote result accepted."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/remote-executor"),
            tools=[
                Tool(
                    "lookup",
                    "Look up a key",
                    {
                        "type": "object",
                        "properties": {"key": {"type": "string"}},
                        "required": ["key"],
                    },
                    local_handler,
                )
            ],
            tool_executor=executor,
        )

        outcome = await runtime.run([AgentMessage.text(Role.USER, "look it up")])

        assert outcome.stop_reason is StopReason.COMPLETE
        assert not local_handler_called
        assert len(executor.calls) == 1  # type: ignore[attr-defined]
        assert executor.calls[0].arguments == {"key": "answer"}  # type: ignore[attr-defined]
        result = next(
            message.result
            for message in runtime.history
            if isinstance(message, ToolResultMessage)
        )
        assert result.content == "remote:ok"
        assert result.metadata == {"backend": "remote"}

    asyncio.run(scenario())


def test_bad_calls_and_ordinary_failures_become_safe_actionable_results() -> None:
    async def scenario() -> None:
        async def explode(arguments: dict[str, object]) -> ToolResult:
            raise RuntimeError("SECRET backend path /private/executor.py")

        schema = {
            "type": "object",
            "properties": {"value": {"type": "integer"}},
            "required": ["value"],
            "additionalProperties": False,
        }
        adapter = ScriptedModelAdapter(
            [
                [
                    ToolCallDelta(0, "bad-json", "explode", "{"),
                    ToolCallDelta(1, "bad-schema", "explode", '{"value": "wrong"}'),
                    ToolCallDelta(2, "unknown", "missing", "{}"),
                    ToolCallDelta(3, "raised", "explode", '{"value": 1}'),
                    ModelEnd(StopReason.TOOL_USE),
                ],
                [TextDelta("I handled the tool errors."), ModelEnd(StopReason.COMPLETE)],
            ]
        )
        runtime = AgentRuntime(
            adapter,
            ModelSpec("scripted/tool-errors"),
            tools=[Tool("explode", "Fail safely", schema, explode)],
        )

        outcome = await runtime.run([AgentMessage.text(Role.USER, "trigger errors")])

        results = [
            message.result
            for message in runtime.history
            if isinstance(message, ToolResultMessage)
        ]
        assert [result.error_code for result in results] == [
            ToolErrorCode.INVALID_JSON,
            ToolErrorCode.INVALID_ARGUMENTS,
            ToolErrorCode.UNKNOWN_TOOL,
            ToolErrorCode.EXECUTION_FAILED,
        ]
        assert all(result.is_error and not result.terminate for result in results)
        assert all("Tool error" in result.content for result in results)
        assert "SECRET" not in "\n".join(result.content for result in results)
        assert outcome.stop_reason is StopReason.COMPLETE

    asyncio.run(scenario())
'''


In [ ]:
OPENAI_TOOL_TEST_SOURCE = r'''from __future__ import annotations

import asyncio
from contextlib import contextmanager
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import json
from threading import Thread
from typing import Iterator

from agent_harness import (
    AgentMessage,
    AgentRuntime,
    ModelSpec,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    StopReason,
    Tool,
    ToolResult,
)


def _chunk(delta: dict[str, object], *, finish_reason: str | None = None) -> dict[str, object]:
    return {
        "id": "chatcmpl-tools",
        "object": "chat.completion.chunk",
        "created": 1,
        "model": "local-tools",
        "choices": [
            {"index": 0, "delta": delta, "finish_reason": finish_reason}
        ],
    }


@contextmanager
def tool_server() -> Iterator[tuple[str, list[dict[str, object]]]]:
    requests: list[dict[str, object]] = []

    class Handler(BaseHTTPRequestHandler):
        def do_POST(self) -> None:
            length = int(self.headers.get("content-length", "0"))
            requests.append(json.loads(self.rfile.read(length)))
            if len(requests) == 1:
                events = [
                    _chunk(
                        {
                            "role": "assistant",
                            "tool_calls": [
                                {
                                    "index": 0,
                                    "id": "call-weather",
                                    "type": "function",
                                    "function": {
                                        "name": "weather",
                                        "arguments": '{"city":"Paris"}',
                                    },
                                }
                            ],
                        }
                    ),
                    _chunk({}, finish_reason="tool_calls"),
                ]
            else:
                events = [
                    _chunk({"role": "assistant", "content": "Paris is clear."}),
                    _chunk({}, finish_reason="stop"),
                ]
            payload = "".join(
                f"data: {json.dumps(event)}\n\n" for event in events
            ) + "data: [DONE]\n\n"
            encoded = payload.encode()
            self.send_response(200)
            self.send_header("content-type", "text/event-stream")
            self.send_header("content-length", str(len(encoded)))
            self.end_headers()
            self.wfile.write(encoded)

        def log_message(self, format: str, *args: object) -> None:
            return

    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        host, port = server.server_address
        yield f"http://{host}:{port}/v1", requests
    finally:
        server.shutdown()
        server.server_close()
        thread.join(timeout=5)


def test_openai_compatible_adapter_translates_tool_definitions_and_results() -> None:
    async def weather(arguments: dict[str, object]) -> ToolResult:
        return ToolResult("clear")

    with tool_server() as (base_url, requests):
        runtime = AgentRuntime(
            OpenAICompatibleAdapter(
                OpenAICompatibleConfig(base_url=base_url, api_key="explicit-key")
            ),
            ModelSpec("local-tools"),
            tools=[
                Tool(
                    "weather",
                    "Get weather",
                    {
                        "type": "object",
                        "properties": {"city": {"type": "string"}},
                        "required": ["city"],
                    },
                    weather,
                )
            ],
        )
        outcome = asyncio.run(
            runtime.run([AgentMessage.text(Role.USER, "weather in Paris")])
        )

    assert outcome.stop_reason is StopReason.COMPLETE
    assert requests[0]["tools"] == [
        {
            "type": "function",
            "function": {
                "name": "weather",
                "description": "Get weather",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                },
            },
        }
    ]
    assert requests[1]["messages"][-1] == {
        "role": "tool",
        "tool_call_id": "call-weather",
        "content": "clear",
    }
'''


## Observable Trace

The Runtime Event stream now brackets each Tool Batch and records each executable call's start and completion. Completion order is observational; it never changes the ordered conversation supplied to the next model turn.

In [ ]:
observable_trace = [
    {
        'sequence': event.sequence,
        'type': event.type.value,
        'tool_call_id': event.tool_call_id,
    }
    for event in minimal_events
]
assert [item['sequence'] for item in observable_trace] == list(
    range(1, len(observable_trace) + 1)
)
assert 'tool_batch_start' in [item['type'] for item in observable_trace]
assert 'tool_call_end' in [item['type'] for item in observable_trace]
observable_trace


## Failure Boundaries and Trade-offs

Malformed JSON, Schema violations, unknown names, and ordinary Tool or executor exceptions become concise `ToolResult` errors so the model can revise its next call. Raw exception text and provider dictionaries stay behind their seams. Cancellation remains distinct and is not swallowed as an ordinary Tool failure.

The default output budget is 50 KB or 2000 lines. Truncation records original extent, retained byte and line ranges, direction, and a complete-output reference. At this pre-persistence Checkpoint, missing references are explicitly unavailable; later durable Runs can supply Artifact references. The core installs no coding Tools, approvals, permission policy, or sandbox claim.

In [ ]:
async def failure_demo():
    async def echo(arguments):
        return chapter3.ToolResult(str(arguments['value']))

    adapter = chapter3.ScriptedModelAdapter([
        [
            chapter3.ToolCallDelta(0, 'bad-json', 'echo', '{'),
            chapter3.ModelEnd(chapter3.StopReason.TOOL_USE),
        ],
        [
            chapter3.TextDelta('Recovered from invalid JSON.'),
            chapter3.ModelEnd(chapter3.StopReason.COMPLETE),
        ],
    ])
    runtime = chapter3.AgentRuntime(
        adapter,
        chapter3.ModelSpec('scripted/failure-demo'),
        tools=[chapter3.Tool(
            'echo',
            'Echo one integer',
            {
                'type': 'object',
                'properties': {'value': {'type': 'integer'}},
                'required': ['value'],
            },
            echo,
        )],
    )
    outcome = await runtime.run([
        chapter3.AgentMessage.text(chapter3.Role.USER, 'demonstrate failure')
    ])
    error_message = next(
        message for message in runtime.history
        if isinstance(message, chapter3.ToolResultMessage)
    )
    return outcome, error_message

failure_outcome, failure_message = await failure_demo()
assert failure_message.result.error_code is chapter3.ToolErrorCode.INVALID_JSON
assert failure_outcome.stop_reason is chapter3.StopReason.COMPLETE


## Checkpoint Export and Verification

Chapter metadata names Chapter 2 as the base Checkpoint. Export carries forward every cumulative source and regression test, replaces the evolved modules, adds Tool tests, writes a deterministic manifest, then compiles, installs without dependency resolution, imports, and runs all tests before publishing Chapter 3.

In [ ]:
PYPROJECT_SOURCE = r'''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "agent-harness"
version = "0.3.0"
description = "Chapter 3 structured Tool loop and deterministic batches"
readme = "README.md"
requires-python = ">=3.11"
dependencies = ["jsonschema>=4.23,<5", "openai>=1.40,<3"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
agent_harness = ["py.typed"]

[tool.pytest.ini_options]
testpaths = ["tests"]
'''


In [ ]:
README_SOURCE = r'''# Agent Harness — Chapter 3 Checkpoint

This cumulative Checkpoint adds explicit structured Tools to the asynchronous Runtime. A Tool declares its name, description, object-shaped JSON Schema, async behavior, concurrency requirement, and output direction. `AgentRuntime` parses and validates model arguments, exposes a stable `PreparedToolCall` preflight value, delegates through a replaceable `ToolExecutor`, and records typed `ToolResultMessage` values before the next model turn.

Read-only Tool batches run concurrently while completion Events reflect actual completion order and conversation history remains in model call order. One sequential or mutating Tool makes the complete batch deterministic and sequential. Invalid JSON, invalid arguments, unknown Tools, and ordinary execution exceptions become safe actionable Tool Errors.

Model-facing output defaults to 50 KB or 2000 lines and carries a structured truncation notice plus a complete-output reference contract when bounded. Automatic continuation stops only when every result in a Tool Batch has `terminate=True`. The core registers no Tool, approval UI, or permission policy implicitly.

All ordinary tests are deterministic and offline. The inherited real-endpoint smoke remains supplementary and explicitly credential-gated.
'''


In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '03_structured_tools.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch03',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '03_structured_tools.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch03',
) == ()
checkpoint_result


## Public API Summary

Declare `Tool(name, description, input_schema, execute, sequential=..., output_direction=...)`. Construct `AgentRuntime(adapter, model, tools=[...], tool_executor=..., tool_output_budget=...)`, then call the unchanged `start` or `run` interface. `PreparedToolCall`, `ToolExecutor`, `ToolResult`, `ToolResultMessage`, `ToolErrorCode`, `ToolOutputBudget`, `TruncationNotice`, and `CompleteOutputReference` are public replaceable or inspectable contracts. With no `tools` argument, Runtime advertises and executes nothing.